# Lab 11 — 몬테카를로와 Bootstrap

**확률통계 · Week 11 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 몬테카를로 오차가 $O(1/\sqrt{n})$ 임을 로그-로그 기울기로 확인한다.
2. **고차원**에서 격자 방식이 무너지고 몬테카를로가 살아남는 것을 본다.
3. **Bootstrap**으로 중앙값의 신뢰구간을 구하고, **언제 무너지는지** 관찰한다.

⏱ **예상 소요 시간: 35분**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. 몬테카를로로 적분하기

$$I = \int_0^1 e^{-x^2}\,dx$$

손으로는 못 푸는 적분이다. **기댓값 → 표본평균**으로 바꾼다.

### 실습 1

In [ ]:
def mc_integral(n, rng):
    u = rng.random(n)
    g = np.exp(-u ** 2)
    # TODO 1: 추정값(표본평균)과 표준오차(std/sqrt(n))를 돌려주세요
    est = 0.0
    se = 0.0
    return est, se


for n in [1000, 100_000, 1_000_000]:
    est, se = mc_integral(n, rng)
    print(f"n={n:>9,}  추정 {est:.5f} +- {1.96 * se:.5f}")

print("\n참값 (scipy 수치적분): 0.746824")

> **추정값만 보지 말고 구간을 보자.** 참값이 구간 안에 들어오는가?
> 몬테카를로 결과는 **항상 오차와 함께** 보고해야 한다.

### 실습 2 — 오차 수렴 속도

In [ ]:
TRUE = 0.7468241328124271
ns = np.unique(np.logspace(2, 6, 25).astype(int))
errs = [abs(mc_integral(n, rng)[0] - TRUE) for n in ns]

plt.figure(figsize=(6.5, 4.2))
plt.loglog(ns, errs, "o", ms=6, label="actual error")
plt.loglog(ns, 0.3 / np.sqrt(ns), lw=2, color="red", label="C/sqrt(n)")
plt.xlabel("n")
plt.ylabel("|estimate - true|")
plt.title("Monte Carlo error")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

# TODO 2: 로그-로그 기울기를 구하세요
#         힌트: np.polyfit(np.log(ns), np.log(np.maximum(errs, 1e-12)), 1)[0]
slope = 0.0
print(f"로그-로그 기울기: {slope:.3f}   (이론 -0.5)")

## Part 2. 고차원에서의 승부

$d$ 차원 단위 초구의 부피 비율(= 정육면체 안에서 반지름 1 구 안에 들어갈 확률)을 구한다.

- **격자 방식**: 축마다 $k$ 개 점 → 전체 $k^d$ 개. $d$ 가 커지면 폭발한다
- **몬테카를로**: 그냥 $n$ 개 뽑는다. $d$ 는 상관없다

### 실습 3

In [ ]:
def mc_ball_fraction(d, n, rng):
    pts = rng.uniform(-1, 1, size=(n, d))
    # TODO 3: 원점에서의 거리 제곱이 1 이하인 점의 비율을 구하세요
    #         힌트: ((pts ** 2).sum(axis=1) <= 1).mean()
    return 0.0


print(f"{'d':>4}{'격자 필요 점수 (축당 100)':>28}{'MC 추정 (n=200,000)':>24}")
for d in [1, 2, 3, 5, 10, 20]:
    grid = 100 ** d
    frac = mc_ball_fraction(d, 200_000, rng)
    grid_str = f"10^{d * 2}" if d > 3 else f"{grid:,}"
    print(f"{d:>4}{grid_str:>28}{frac:>24.5f}")

🤯 두 가지가 동시에 보인다.

1. **격자 방식은 $d=10$ 만 되어도 $10^{20}$ 개** — 계산 불가능. 몬테카를로는 20만 개로 끝
2. **차원이 오르면 구의 부피 비율이 0으로 간다** — 고차원 정육면체는 "부피가 구석에 몰려 있다"

두 번째 현상도 **차원의 저주**의 한 얼굴이다. 고차원 공간에 대한 우리의 직관은 대부분 틀린다.

## Part 3. Bootstrap

이제 도구를 바꾼다. **표본 하나로 불확실성을 재는** 방법이다.

### 실습 4 — 중앙값의 신뢰구간

In [ ]:
data = rng.lognormal(mean=1.0, sigma=0.8, size=60)
print(f"표본 크기 {len(data)}, 관측 중앙값 {np.median(data):.4f}")

B = 5000
# TODO 4: 복원추출로 재표본을 만들고 중앙값을 B번 계산하세요
#         힌트: np.median(rng.choice(data, size=len(data), replace=True))
boot = np.array([np.median(data) for _ in range(B)])

lo, hi = np.percentile(boot, [2.5, 97.5])

plt.figure(figsize=(7, 4))
plt.hist(boot, bins=50, alpha=0.85)
plt.axvline(np.median(data), color="black", lw=2, label="observed median")
plt.axvline(lo, color="red", ls="--", lw=2, label="95% CI")
plt.axvline(hi, color="red", ls="--", lw=2)
plt.xlabel("bootstrap median")
plt.ylabel("count")
plt.title("Bootstrap distribution of the median")
plt.legend()
plt.show()

print(f"bootstrap 표준오차 {boot.std():.4f}")
print(f"95% 신뢰구간 [{lo:.4f}, {hi:.4f}]")

> ⚠️ **`replace=True` 가 핵심이다.** 이걸 빠뜨리면 매번 같은 표본이 나와
> 표준오차가 0이 되어버린다. (위 배포용 코드의 초기값이 정확히 그 상태다)

### 실습 5 — Bootstrap은 언제 무너지는가

표본 크기를 줄여가며 신뢰구간이 어떻게 되는지 본다.
**참값**을 알고 있으므로(우리가 만든 데이터니까) 구간이 참값을 덮는지 확인할 수 있다.

In [ ]:
TRUE_MEDIAN = np.exp(1.0)
print(f"참 중앙값 = {TRUE_MEDIAN:.4f}\n")

for n in [10, 30, 100, 500]:
    sample = rng.lognormal(mean=1.0, sigma=0.8, size=n)
    boot = np.array([np.median(rng.choice(sample, size=n, replace=True))
                     for _ in range(2000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    # TODO 5: 참값이 구간 안에 들어오는지 판정하세요.  힌트: lo <= TRUE_MEDIAN <= hi
    covered = False
    print(f"n={n:>4}  구간 [{lo:6.3f}, {hi:6.3f}]  폭 {hi - lo:5.3f}  "
          f"참값 포함: {'O' if covered else 'X'}")

🤔 **$n$ 이 커질수록 구간이 좁아진다.** 폭이 대략 $1/\sqrt{n}$ 로 줄어드는지 확인해보자.

($n$ 이 50배 커지면 폭은 약 $\sqrt{50} \approx 7$배 좁아져야 한다)

$n=10$ 의 구간을 보자. 참값을 덮기는 했지만 **폭이 3을 넘는다** —
"중앙값은 1.5와 4.9 사이 어딘가"라는 말은 사실상 아무 정보도 주지 않는다.

**`size=10` 부분의 시드를 바꿔가며 여러 번 돌려보자.** $n=10$ 에서는 참값을
**놓치는 경우도 나온다.** 원래 표본이 우연히 치우치면 bootstrap은 그 치우침을 그대로 물려받는다.

> **Bootstrap은 데이터에 없는 정보를 만들어내지 못한다.**
> 다만 "내가 가진 데이터로 이만큼밖에 말할 수 없다"를 **정직하게** 알려준다.

---

## 마무리 — 자가 점검

- [ ] 적분을 기댓값으로 바꿔 몬테카를로로 풀 수 있다
- [ ] 몬테카를로 결과에 신뢰구간을 붙일 수 있다
- [ ] 고차원에서 격자 방식이 왜 무너지는지 설명할 수 있다
- [ ] Bootstrap으로 임의의 통계량의 신뢰구간을 구할 수 있다
- [ ] `replace=True` 를 빠뜨리면 안 되는 이유를 안다

**오늘 배운 것을 한 문장으로.**

> (여기에 작성)

### 📌 미니 프로젝트 2 — `hw/mini_project_2.md` (W15 제출). 오늘 배운 bootstrap이 핵심 도구다